# 03_bronze_to_silver
Reads Bronze rows for one date, casts and cleans them, sends bad rows to silver_quarantine, and MERGEs the rest into silver_steam on appid.
Parameters: date (YYYY-MM-DD), batch_id (optional, to reprocess a single Bronze batch).

In [0]:
dbutils.widgets.text("date", "")
dbutils.widgets.text("batch_id", "")

In [0]:
import uuid
from datetime import datetime, date as _date
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

CATALOG, SCHEMA = "workspace", "steam"
LAYER = "Bronze-to-Silver"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

run_date = dbutils.widgets.get("date").strip() or str(_date.today())
datetime.strptime(run_date, "%Y-%m-%d")  # fails fast on a bad date
batch_filter = dbutils.widgets.get("batch_id").strip()
run_id = str(uuid.uuid4())
print(run_date, batch_filter or "(all batches)", run_id)

In [0]:
SOURCE_COLS = ["appid", "name", "developer", "publisher", "score_rank", "userscore",
               "positive", "negative", "owners", "average_forever", "average_2weeks",
               "median_forever", "median_2weeks", "price", "initialprice", "discount", "ccu"]

def read_bronze():
    df = spark.table("bronze_steam").where(F.col("load_date") == F.to_date(F.lit(run_date)))
    if batch_filter:
        df = df.where(F.col("batch_id") == batch_filter)
    return df

print(read_bronze().count())

In [0]:
def cast_columns(df):
    """Cast every raw string to its Silver type. A failed cast gives NULL instead of an error."""
    owners_clean = "replace(owners, ',', '')"
    return (df
        .withColumn("c_appid", F.expr("try_cast(appid as int)"))
        .withColumn("c_positive", F.expr("try_cast(positive as int)"))
        .withColumn("c_negative", F.expr("try_cast(negative as int)"))
        .withColumn("c_ccu", F.expr("try_cast(ccu as int)"))
        .withColumn("c_price", F.expr("try_cast(price as double) / 100"))
        .withColumn("c_initialprice", F.expr("try_cast(initialprice as double) / 100"))
        .withColumn("c_discount", F.expr("try_cast(discount as double)"))
        .withColumn("c_owners_min", F.expr(f"try_cast(regexp_extract({owners_clean}, '^([0-9]+)', 1) as bigint)"))
        .withColumn("c_owners_max", F.expr(f"try_cast(regexp_extract({owners_clean}, '([0-9]+) *$', 1) as bigint)")))

display(cast_columns(read_bronze())
        .select("appid", "owners", "c_owners_min", "c_owners_max", "price", "c_price", "c_ccu")
        .limit(5))

In [0]:
def add_reject_reason(df):
    """reject_reason stays NULL for good rows and lists every failed check for bad ones."""
    def bad(raw, cast):  # raw has a value but the cast failed
        return F.col(raw).isNotNull() & (F.trim(F.col(raw)) != "") & F.col(cast).isNull()

    checks = [
        F.when(F.col("c_appid").isNull(), F.lit("invalid appid")),
        F.when(bad("positive", "c_positive"), F.lit("uncastable positive")),
        F.when(bad("negative", "c_negative"), F.lit("uncastable negative")),
        F.when(bad("ccu", "c_ccu"), F.lit("uncastable ccu")),
        F.when(bad("price", "c_price"), F.lit("uncastable price")),
        F.when(bad("initialprice", "c_initialprice"), F.lit("uncastable initialprice")),
        F.when(bad("discount", "c_discount"), F.lit("uncastable discount")),
        F.when(bad("owners", "c_owners_min") | bad("owners", "c_owners_max"), F.lit("unparsable owners")),
    ]
    reason = F.concat_ws("; ", *checks)
    return df.withColumn("reject_reason", F.when(reason != "", reason))

display(add_reject_reason(cast_columns(read_bronze())).groupBy("reject_reason").count())

In [0]:
def build_silver(df):
    """Good rows only: drop dead entries, keep one row per appid, add derived columns."""
    w = Window.partitionBy("c_appid").orderBy(F.col("load_timestamp").desc())
    good = (df.where(F.col("reject_reason").isNull())
              .withColumn("total_reviews", F.coalesce("c_positive", F.lit(0)) + F.coalesce("c_negative", F.lit(0)))
              .where(~((F.col("total_reviews") == 0) & (F.coalesce("c_ccu", F.lit(0)) == 0)))
              .withColumn("rn", F.row_number().over(w)).where("rn = 1"))

    def clean(c):  # trim text, turn empty strings into NULL
        return F.when(F.trim(F.col(c)) != "", F.trim(F.col(c)))

    return good.select(
        F.col("c_appid").alias("appid"),
        clean("name").alias("name"),
        clean("developer").alias("developer"),
        clean("publisher").alias("publisher"),
        F.col("c_positive").alias("positive"),
        F.col("c_negative").alias("negative"),
        "total_reviews",
        F.expr("try_divide(c_positive, total_reviews)").alias("approval_rate"),
        F.col("c_ccu").alias("ccu"),
        F.col("c_owners_min").alias("owners_min"),
        F.col("c_owners_max").alias("owners_max"),
        F.col("c_price").alias("price_usd"),
        F.col("c_initialprice").alias("initialprice_usd"),
        F.col("c_discount").alias("discount_pct"),
        (F.col("c_price") == 0).alias("is_free"),
        "row_hash",
        "load_date",
        F.current_timestamp().alias("load_timestamp"))

silver_preview = build_silver(add_reject_reason(cast_columns(read_bronze())))
print(silver_preview.count())
display(silver_preview.limit(5))

In [0]:
def write_quarantine(df):
    """Send rejected rows to silver_quarantine. MERGE keeps a re-run from duplicating them."""
    bad = (df.where(F.col("reject_reason").isNotNull())
             .select(*SOURCE_COLS, "reject_reason", "load_date",
                     F.current_timestamp().alias("load_timestamp"))
             .dropDuplicates(["appid", "reject_reason"]))
    n = bad.count()
    if n:
        bad.createOrReplaceTempView("quarantine_updates")
        spark.sql("""
            MERGE INTO silver_quarantine t
            USING quarantine_updates s
            ON t.appid <=> s.appid AND t.load_date = s.load_date AND t.reject_reason = s.reject_reason
            WHEN NOT MATCHED THEN INSERT *
        """)
    return n

In [0]:
DATA_COLS = ["name", "developer", "publisher", "positive", "negative", "total_reviews",
             "approval_rate", "ccu", "owners_min", "owners_max", "price_usd",
             "initialprice_usd", "discount_pct", "is_free", "row_hash"]

def merge_into_silver(df):
    """Insert new appids; update an existing one only if its hash changed and the data isn't older."""
    df.createOrReplaceTempView("silver_updates")
    set_clause = ", ".join(f"t.{c} = s.{c}" for c in DATA_COLS)
    insert_cols = ", ".join(["appid"] + DATA_COLS + ["first_load_date", "last_load_date", "load_timestamp"])
    insert_vals = ", ".join([f"s.{c}" for c in ["appid"] + DATA_COLS] + ["s.load_date", "s.load_date", "s.load_timestamp"])
    res = spark.sql(f"""
        MERGE INTO silver_steam t
        USING silver_updates s
        ON t.appid = s.appid
        WHEN MATCHED AND t.row_hash <> s.row_hash AND s.load_date >= t.last_load_date THEN UPDATE SET
            {set_clause}, t.last_load_date = s.load_date, t.load_timestamp = s.load_timestamp
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
    """).first()
    return res["num_inserted_rows"], res["num_updated_rows"]

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("file_processed", StringType()),
    StructField("batch_id", StringType()), StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()), StructField("status", StringType()),
    StructField("rows_inserted", LongType()), StructField("rows_updated", LongType()),
    StructField("rows_skipped_unchanged", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType())])

def write_log(start, status, load_type=None, inserted=None, updated=None,
              skipped=None, quarantined=None, error=None):
    param = f"bronze_steam load_date={run_date}" + (f" batch_id={batch_filter}" if batch_filter else "")
    row = (str(uuid.uuid4()), LAYER, load_type, param, batch_filter or None, start, datetime.now(),
           status, inserted, updated, skipped, quarantined, error)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable("pipeline_execution_logs")

In [0]:
start = datetime.now()
load_type = None
try:
    bronze = read_bronze()
    types = [r.load_type for r in bronze.select("load_type").distinct().collect()]
    load_type = ",".join(sorted(types)) if types else None

    checked = add_reject_reason(cast_columns(bronze))
    quarantined = write_quarantine(checked)

    silver_rows = build_silver(checked)
    candidates = silver_rows.count()
    inserted, updated = merge_into_silver(silver_rows)
    skipped = candidates - inserted - updated

    write_log(start, "Success", load_type, inserted, updated, skipped, quarantined)
    print(f"candidates: {candidates} | inserted: {inserted} | updated: {updated} | "
          f"skipped: {skipped} | quarantined: {quarantined}")
except Exception as e:
    write_log(start, "Failure", load_type, error=str(e)[:2000])
    raise

In [0]:
display(spark.sql("SELECT COUNT(*) AS silver_rows, COUNT(DISTINCT appid) AS distinct_appids FROM silver_steam"))
display(spark.sql("SELECT appid, COUNT(*) AS n FROM silver_steam GROUP BY appid HAVING COUNT(*) > 1"))
display(spark.sql("SELECT layer, load_type, file_processed, status, rows_inserted, rows_updated, rows_skipped_unchanged, rows_quarantined, start_time FROM pipeline_execution_logs ORDER BY start_time DESC LIMIT 5"))